In [ ]:
# SET UP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne
import os
import mne
import traceback
import gc

In [ ]:
# Paths and constants
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

event_id = {'Wd2N': 1, 'Wd2E': 2}  # Normal and slowed
tmin, tmax = -0.1, 0.8  # Time window in seconds
baseline = (tmin, 0)  # Baseline correction period

def create_and_save_epochs(participant):
    participant_path = os.path.join(root_dir, participant)
    raw_file = os.path.join(participant_path, f"{participant}_ica_reref_raw.fif")
    epo_file = os.path.join(participant_path, f"{participant}_epo.fif")

    if not os.path.exists(raw_file):
        print(f"[{participant}] ❌ ICA reref file not found.")
        return

    if os.path.exists(epo_file):
        print(f"[{participant}] ✅ Epochs already exist, skipping.")
        return

    try:
        print(f"[{participant}] Creating epochs...")

        raw = mne.io.read_raw_fif(raw_file, preload=True)
        raw.set_montage('GSN-HydroCel-128')

        # Convert annotations to events
        events, _ = mne.events_from_annotations(raw, event_id=event_id)

        # Epoching
        epochs = mne.Epochs(
            raw,
            events,
            event_id=event_id,
            tmin=tmin,
            tmax=tmax,
            baseline=baseline,
            reject_by_annotation=True,
            preload=True
        )

        # Save epochs
        epochs.save(epo_file, overwrite=True)
        print(f"[{participant}] ✅ Epochs saved to {epo_file}")

        del raw, epochs
        gc.collect()

    except Exception as e:
        print(f"[{participant}] ⚠️ Error during epoching: {e}")
        traceback.print_exc()

# Run for all participants
for pid in participants:
    create_and_save_epochs(pid)

In [ ]:
import os
import mne

# Set your path
participant = '10'  # change to any subject number
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
epo_path = os.path.join(root_dir, participant, f"{participant}_epo.fif")

# Load epochs
epochs = mne.read_epochs(epo_path, preload=True)
print(epochs)

# Check the event counts
print("Event count:")
print(epochs.event_id)
print(epochs.selection.shape)

# Plot all events
epochs.plot(n_epochs=10, block=True)  # interactively scroll through trials